# M2.S1 — Parallel Thinking and Decomposition
## Interactive HPC notebook

This notebook accompanies **M2.S1 — Parallel Thinking and Decomposition**.

The goal is **not** to learn OpenMP or MPI syntax yet. Instead, you will run small experiments that make the main ideas of the session visible:

1. **Independence creates parallelism**
2. **Task and data decomposition expose different kinds of work**
3. **Dependencies remove concurrency**
4. **Synchronization can force waiting**
5. **Granularity matters**
6. **Load balance matters**
7. **Logical work is mapped to physical resources by the scheduler**

### Recommended execution
- **IE/SciTech JupyterHub and HPC cluster:** run the complete notebook.
- **Local Jupyter / Colab:** conceptual Python activities work; skip Slurm cells if Slurm is unavailable.

The short Python timing examples use small `sleep()` calls to simulate work. They are conceptual experiments, **not hardware benchmarks**.

> **Notebook build: M2S1-2026-09-20-v2**
>
> ## How to run this notebook
>
> **During class:** run each code cell with **Shift+Enter** when the slide says to test the concept.
>
> **For instructor validation:** use **Kernel → Restart Kernel and Run All Cells** (or the JupyterLab “run all” button). Wait until the final code cell finishes.
>
> A code cell showing **`In [ ]`** has **not been executed yet**. It will not display output until you run it.
>
> The notebook intentionally does **not** store precomputed outputs, so students can predict before running the experiment.

## Before you start — predict first

For every exercise:

> **PREDICT → RUN → OBSERVE → EXPLAIN**

Do not run the cell immediately. First discuss what you expect to happen and why.


**Then run the code cell after making your prediction.**

In [ ]:
import time
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier


# 0 — Where am I running?

Before the parallel-thinking experiments, identify the environment.

### Predict
- What hostname do you expect?
- Is Slurm available from this Jupyter environment?
- Is opening a notebook the same as obtaining compute resources?

In [ ]:
import os
import platform

print('Host:', platform.node())
print('User:', os.environ.get('USER', 'unknown'))
print('CPU count visible to Python:', os.cpu_count())
print('sinfo available:', shutil.which('sinfo') is not None)
print('srun available:', shutil.which('srun') is not None)

### Observe and explain
- Note the hostname and visible CPU count.
- Does this prove that your notebook has reserved all those CPUs?
- Why do we still need Slurm for real cluster resource allocation?

# Exercise 1 — Independent work vs. a dependency chain
### Real context: satellite-image preprocessing

A remote-sensing team receives **16 satellite images**. Each image must be independently preprocessed before it is used by an AI model.

First, assume every image is independent.

### Predict
1. With 8 workers, can several images be processed at the same time?
2. Roughly how many waves of work are needed for 16 equal images on 8 workers?
3. What changes if Image `i` requires the result from Image `i-1`?

**Session concept:** *Hardware does not create parallelism; independence in the problem does.*


In [ ]:
N_IMAGES = 16
N_WORKERS = 8
IMAGE_WORK = 0.15  # scaled simulated time per image

def process_image(image_id):
    time.sleep(IMAGE_WORK)
    return image_id

start = time.perf_counter()
for i in range(N_IMAGES):
    process_image(i)
sequential_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    list(pool.map(process_image, range(N_IMAGES)))
parallel_time = time.perf_counter() - start

print(f"Sequential time:           {sequential_time:.2f} s")
print(f"8-worker independent time: {parallel_time:.2f} s")
print(f"Observed speedup:           {sequential_time/parallel_time:.1f}x")


### Now add a dependency

Imagine that each step needs the result of the previous step:

`Image 0 → Image 1 → Image 2 → ...`

This is deliberately artificial for images, but it makes the effect of a dependency visible.


In [ ]:
def dependent_step(previous_result, step_id):
    time.sleep(IMAGE_WORK)
    return previous_result + 1

start = time.perf_counter()
result = 0
for i in range(8):
    result = dependent_step(result, i)
dependency_time = time.perf_counter() - start

print(f"8 dependent steps: {dependency_time:.2f} s")
print("The next step is not ready until the previous one finishes.")


### Explain
- Why did the independent case benefit from several workers?
- Why did the dependency chain not?
- Was the limitation caused by slow hardware or by the structure of the problem?

**Expected idea:** the dependency chain has very little available concurrency.


# 2 — Task decomposition or data decomposition?
### Slide connection: different work or different data?

Before running anything, classify these:

1. Process four independent input files.
2. Split a 4000 × 4000 matrix by rows.
3. Run 100 independent simulation configurations.
4. Apply the same filter to four image regions.

### Predict
Which are most naturally **task decomposition**, which are **data decomposition**, and where could more than one design be reasonable?

In [ ]:
# TASK decomposition: different operations on the same conceptual input
def statistics(x):
    time.sleep(0.20)
    return ('statistics', sum(x) / len(x))

def feature_detection(x):
    time.sleep(0.20)
    return ('features', max(x) - min(x))

def fft_like(x):
    time.sleep(0.20)
    return ('fft-like', sum(v*v for v in x))

data = list(range(100))
with ThreadPoolExecutor(max_workers=3) as pool:
    task_results = list(pool.map(lambda f: f(data),
                                 [statistics, feature_detection, fft_like]))

print('TASK decomposition:')
for result in task_results:
    print(' ', result)

In [ ]:
# DATA decomposition: the same operation on different pieces of data
tiles = [list(range(i*25, (i+1)*25)) for i in range(4)]

def same_filter(tile):
    time.sleep(0.20)
    return sum(v * 2 for v in tile)

with ThreadPoolExecutor(max_workers=4) as pool:
    data_results = list(pool.map(same_filter, tiles))

print('DATA decomposition:', data_results)

### Observe
- In the first example, workers perform **different work**.
- In the second, workers perform the **same work on different data**.

### Try it
Change the number of image tiles from 4 to 8. Does the **type** of decomposition change?

### Explain
Why is decomposition often a **design choice**, rather than one predetermined answer?

# 3 — Dependencies decide what can start
### Slide connection: task graph A → C → E and A+B → D

Use the task graph from the lecture:

- A: no dependency
- B: no dependency
- C: depends on A
- D: depends on A and B
- E: depends on C

### Predict
1. What can run at time 0?
2. After A finishes, what becomes ready?
3. If A is finished but B is not, can D run?
4. Is the waiting caused by slow hardware?

In [ ]:
tasks = {
    'A': set(),
    'B': set(),
    'C': {'A'},
    'D': {'A', 'B'},
    'E': {'C'},
}

def ready_tasks(done):
    return sorted(
        task for task, deps in tasks.items()
        if task not in done and deps.issubset(done)
    )

for done in [set(), {'A'}, {'A', 'B'}, {'A', 'C'}, {'A', 'B', 'C'}]:
    print(f'done={sorted(done)} -> ready={ready_tasks(done)}')

### Observe and explain
- A task becomes ready only when all required inputs/results exist.
- Why does each dependency remove some possible concurrency?
- Would a faster processor make D ready before B finishes?

# 4 — Granularity: too coarse, useful, too fine
### Real context: splitting a large image into tiles

Suppose an image contains a fixed amount of useful work. We can divide it into:

- **2 large tiles** — too coarse for 8 workers
- **8 medium tiles** — enough work to keep 8 workers busy
- **64 tiny tiles** — more parallel pieces, but each task has management overhead

We simulate a small fixed overhead for every task.

### Predict
Which case will finish fastest?


In [ ]:
N_WORKERS = 8
TOTAL_USEFUL_WORK = 3.2
TASK_OVERHEAD = 0.02

def tile_task(useful_work):
    time.sleep(TASK_OVERHEAD)
    time.sleep(useful_work)

def run_granularity(n_tasks):
    useful_per_task = TOTAL_USEFUL_WORK / n_tasks
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        list(pool.map(tile_task, [useful_per_task] * n_tasks))
    return time.perf_counter() - start

for n_tasks in [2, 8, 64]:
    elapsed = run_granularity(n_tasks)
    label = {2: "TOO COARSE", 8: "USEFUL", 64: "TOO FINE"}[n_tasks]
    print(f"{n_tasks:>2} tasks | {label:<10} | elapsed = {elapsed:.2f} s")


### Explain
- Why are 2 tasks inefficient on 8 workers?
- Why can 64 tiny tasks be slower than 8 useful tasks?
- What would happen if task overhead became larger?

**Key idea:** *Large enough to do useful work; small enough to keep workers busy.*


# 5 — Load balancing: same work, different finish time
### Real context: 8 independent simulation jobs

Eight independent simulation cases take:

**8, 7, 6, 5, 4, 3, 2, 1 time units**

You have **4 workers**.

### Poor assignment
- W1: 8 + 7
- W2: 6 + 5
- W3: 4 + 3
- W4: 2 + 1

### Balanced assignment
- W1: 8 + 1
- W2: 7 + 2
- W3: 6 + 3
- W4: 5 + 4

### Predict
Both assignments perform the same total amount of work. Will they finish at the same time?


In [ ]:
SCALE = 0.10

poor = [[8, 7], [6, 5], [4, 3], [2, 1]]
balanced = [[8, 1], [7, 2], [6, 3], [5, 4]]

def run_worker(worker_id, tasks):
    for duration in tasks:
        time.sleep(duration * SCALE)
    return worker_id, sum(tasks)

def run_assignment(assignments):
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = [pool.submit(run_worker, i + 1, tasks)
                   for i, tasks in enumerate(assignments)]
        results = [f.result() for f in futures]
    return time.perf_counter() - start, results

poor_time, poor_results = run_assignment(poor)
balanced_time, balanced_results = run_assignment(balanced)

print("Poor assignment worker loads:", poor_results)
print(f"Poor assignment elapsed:      {poor_time:.2f} s")
print()
print("Balanced worker loads:", balanced_results)
print(f"Balanced assignment elapsed:  {balanced_time:.2f} s")


### Explain
- Which worker determines the completion time in the poor assignment?
- Did the total amount of work change?
- What improved?

**Key idea:** *The slowest worker sets the pace when the next phase must wait for everyone.*


## Optional extension — Dynamic scheduling with unpredictable runtimes
### Context: 20 DNA samples, 4 analysis workers

A genomics lab processes samples through the same pipeline:

**Read data → Quality check → Sequence analysis → Result**

Most samples are quick, but a few take much longer.

Compare:

1. **Static assignment** — each worker receives a fixed group at the beginning.
2. **Dynamic assignment** — whenever a worker finishes, it takes the next available sample.

### Predict
If several long samples are assigned to the same worker, what happens to the others?


In [ ]:
sample_times = [0.50, 0.50, 0.50, 0.50] + [0.10] * 16
samples = [(f"sample_{i+1:02d}", t) for i, t in enumerate(sample_times)]

def analyze_sample(sample):
    sample_id, duration = sample
    time.sleep(duration)
    return sample_id, duration

static_groups = [samples[i:i+5] for i in range(0, len(samples), 5)]

def static_worker(group):
    for sample in group:
        analyze_sample(sample)
    return sum(t for _, t in group)

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    static_loads = list(pool.map(static_worker, static_groups))
static_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(analyze_sample, samples))
dynamic_time = time.perf_counter() - start

print("Static worker loads:", [round(x, 2) for x in static_loads])
print(f"Static elapsed:  {static_time:.2f} s")
print(f"Dynamic elapsed: {dynamic_time:.2f} s")


### Explain
- Why is this workload naturally parallel?
- Is the natural decomposition primarily task or data decomposition?
- Why can equal numbers of samples still produce unequal amounts of work?
- Why does taking a new sample when a worker becomes free improve load balance?

**Key idea:** *Strong parallelism does not automatically guarantee good load balance.*


# 6 — Synchronization: computation, exchange, barrier
### Real context: neighbouring regions in a weather model

A climate/weather domain can be divided into neighbouring regions. Each worker:

1. updates its local region,
2. exchanges boundary information,
3. waits at a synchronization point before the next timestep.

One worker below is deliberately slower.

### Predict
Which worker determines when the next timestep can begin?


In [ ]:
barrier = Barrier(4)
local_compute = [0.20, 0.20, 0.40, 0.20]
boundary_exchange = 0.05

def climate_worker(worker_id):
    start = time.perf_counter()
    time.sleep(local_compute[worker_id])
    compute_done = time.perf_counter() - start
    time.sleep(boundary_exchange)
    before_barrier = time.perf_counter() - start
    barrier.wait()
    finished = time.perf_counter() - start
    return worker_id + 1, compute_done, before_barrier, finished

with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(climate_worker, range(4)))

for wid, compute_done, before_barrier, finished in sorted(results):
    print(
        f"W{wid}: compute done {compute_done:.2f}s | "
        f"barrier arrival {before_barrier:.2f}s | "
        f"continued {finished:.2f}s"
    )


### Explain
- Which workers wait?
- Why is waiting required?
- What would happen if the grid were divided into many more, much smaller regions?

**Key idea:** *A decomposition can be parallel but not independent.*


# 7 — From logical tasks to real HPC resources
### Cluster-only demo

Your program/problem defines **logical tasks**. The HPC scheduler allocates physical resources.

Before running, predict:

> If we request 8 tasks, did our algorithm decide which physical server Task 3 will run on?


In [ ]:
print("Checking for Slurm...")
if shutil.which("sinfo") is None:
    print("Slurm commands are not available here.")
    print("Run this section on the IE HPC/JupyterHub environment.")
else:
    print("\n--- hostname ---")
    subprocess.run(["hostname"])
    print("\n--- user ---")
    subprocess.run(["whoami"])
    print("\n--- Slurm resources ---")
    subprocess.run(["sinfo"])


In [ ]:
if shutil.which("srun") is None:
    print("srun not found — skip this cell outside the HPC cluster.")
else:
    cmd = '''srun -n 8 bash -lc 'echo "task=$SLURM_PROCID host=$(hostname)"' '''
    print("Running:", cmd)
    subprocess.run(cmd, shell=True)


### Observe and explain
- How many logical task IDs did Slurm create?
- Did all tasks run on the same hostname or on more than one?
- Did your Python algorithm choose those hostnames?

**Answer:** the decomposition/application defines the logical work; **Slurm decides the physical placement** according to the resources allocated by the cluster.

### Optional two-node demonstration
Only try this if `sinfo` shows at least two suitable nodes available:

```bash
srun -N 2 -n 8 --ntasks-per-node=4 bash -lc 'echo "task=$SLURM_PROCID host=$(hostname)"'
```

If the teaching cluster requires a partition, add the appropriate `-p <partition>` option.


# Challenge — Apply the parallel-thinking checklist

Choose one real scenario:

- satellite-image preprocessing,
- climate simulation,
- DNA-sample analysis,
- matrix processing.

In pairs, answer:

1. What are the units of work?
2. Is the starting point task decomposition, data decomposition, or a mixture?
3. Which units can run independently?
4. What dependencies exist?
5. What must be communicated?
6. Where might workers synchronize?
7. Could load imbalance or poor granularity occur?
8. What cluster resources would you request?

Be ready to explain your design in **60 seconds**.

## What did we learn?

1. **Parallelism comes from independence.**
2. **Task decomposition = different work; data decomposition = same work on different data.**
3. **Dependencies decide when work becomes ready.**
4. **Synchronization preserves correctness but can create waiting.**
5. **Granularity and load balance determine how well workers are used.**
6. **The application defines logical work; Slurm maps it to physical resources.**

## Next
In **M2.S2 — OpenMP**, we will express shared-memory parallelism in code.